Shared Blocks across the architectures like Multi-Headed Attention, Positional Encoding and Attention masks

In [1]:

import math
import random
import torch
import torch.nn as nn
import torch.nn.functional as F

random.seed(42)
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_length=64):
        super().__init__()

        position = torch.arange(max_length).float().unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float()
            * (-math.log(10000.0) / d_model)
        )

        pe = torch.zeros(max_length, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=32, num_heads=4):
        super().__init__()
        assert d_model % num_heads == 0

        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, query, key, value, mask=None):
        B, Tq, D = query.shape
        Tk = key.size(1)

        def split_heads(x, length):
            return x.reshape(
                B, length, self.num_heads, self.head_dim
            ).transpose(1, 2)

        Q = split_heads(self.q_proj(query), Tq)
        K = split_heads(self.k_proj(key), Tk)
        V = split_heads(self.v_proj(value), Tk)

        scores = Q @ K.transpose(-2, -1)
        scores = scores / math.sqrt(self.head_dim)

        if mask is not None:
            scores = scores.masked_fill(~mask, float("-inf"))

        weights = torch.softmax(scores, dim=-1)
        context = weights @ V

        context = context.transpose(1, 2).contiguous()
        context = context.reshape(B, Tq, D)

        return self.out_proj(context)


class EncoderBlock(nn.Module):
    def __init__(self, d_model=32, num_heads=4, d_ff=64):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, num_heads)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        x = self.norm1(x + self.attention(x, x, x))
        x = self.norm2(x + self.ffn(x))
        return x


class DecoderBlock(nn.Module):
    def __init__(self, d_model=32, num_heads=4, d_ff=64):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)

        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

    def forward(self, x, encoder_output, causal_mask):
        # Causal self-attention over generated tokens
        x = self.norm1(
            x + self.self_attn(x, x, x, causal_mask)
        )

        # Cross-attention to the encoder's output
        x = self.norm2(
            x + self.cross_attn(x, encoder_output, encoder_output)
        )

        x = self.norm3(x + self.ffn(x))
        return x


def causal_mask(length, device):
    # True means the position is allowed to attend
    return torch.tril(
        torch.ones(length, length, dtype=torch.bool, device=device)
    )[None, None, :, :]


def make_vocab(sentences, special_tokens=()):
    words = sorted({
        word for sentence in sentences for word in sentence.split()
    })
    vocab = {token: i for i, token in enumerate(special_tokens)}
    for word in words:
        if word not in vocab:
            vocab[word] = len(vocab)
    return vocab


def encode(sentence, vocab):
    return [vocab[word] for word in sentence.split()]


Using: cpu


Train a small sentiment analysis model using Encoder Only

In [2]:
# Small training dataset
sentences = [
    "this movie is good",
    "this film is great",
    "this movie is excellent",
    "this film is wonderful",
    "this movie is bad",
    "this film is awful",
    "this movie is terrible",
    "this film is boring",
]

labels = torch.tensor(
    [1, 1, 1, 1, 0, 0, 0, 0],
    dtype=torch.long,
    device=device
)

vocab = make_vocab(sentences)
X = torch.tensor(
    [encode(s, vocab) for s in sentences],
    dtype=torch.long,
    device=device
)

class EncoderClassifier(nn.Module):
    def __init__(self, vocab_size, num_classes=2):
        super().__init__()
        d_model = 32

        self.embedding = nn.Embedding(vocab_size, d_model)
        self.position = PositionalEncoding(d_model)
        self.encoder = nn.ModuleList([
            EncoderBlock(d_model) for _ in range(2)
        ])
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, token_ids):
        x = self.position(self.embedding(token_ids))

        for layer in self.encoder:
            x = layer(x)

        return self.classifier(x.mean(dim=1))


encoder_model = EncoderClassifier(len(vocab)).to(device)
optimizer = torch.optim.Adam(encoder_model.parameters(), lr=0.01)

for epoch in range(201):
    encoder_model.train()
    optimizer.zero_grad()

    logits = encoder_model(X)
    loss = F.cross_entropy(logits, labels)

    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print(f"Epoch {epoch:3d} | Loss: {loss.item():.4f}")


# Test on unseen combinations of known words
encoder_model.eval()
test_sentences = ["this movie is wonderful", "this film is terrible"]
test_X = torch.tensor(
    [encode(s, vocab) for s in test_sentences],
    dtype=torch.long,
    device=device
)

with torch.no_grad():
    predictions = encoder_model(test_X).argmax(dim=-1).tolist()

class_names = ["negative", "positive"]

for sentence, prediction in zip(test_sentences, predictions):
    print(f"{sentence:30s} -> {class_names[prediction]}")


Epoch   0 | Loss: 0.7206
Epoch  50 | Loss: 0.0005
Epoch 100 | Loss: 0.0002
Epoch 150 | Loss: 0.0001
Epoch 200 | Loss: 0.0001
this movie is wonderful        -> positive
this film is terrible          -> negative


Train a Seq generation model using Decoder Only architecture

In [4]:

# Each sequence contains exactly three words
sequences = [
    "i like cats",
    "you like dogs",
    "i love music",
    "you love books",
    "cats chase mice",
    "dogs chase balls",
]

special = ["<PAD>", "<SOS>", "<EOS>", "<UNK>"]
lm_vocab = make_vocab(sequences, special)
id_to_word = {i: word for word, i in lm_vocab.items()}

# Add start and end tokens
def encode_lm(sentence):
    return [
        lm_vocab["<SOS>"],
        *encode(sentence, lm_vocab),
        lm_vocab["<EOS>"]
    ]

lm_data = torch.tensor(
    [encode_lm(s) for s in sequences],
    dtype=torch.long,
    device=device
)

class CausalBlock(nn.Module):
    def __init__(self, d_model=32, num_heads=4, d_ff=64):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, num_heads)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        mask = causal_mask(x.size(1), x.device)
        x = self.norm1(x + self.attention(x, x, x, mask))
        x = self.norm2(x + self.ffn(x))
        return x


class DecoderOnlyLM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        d_model = 32
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.position = PositionalEncoding(d_model)
        self.blocks = nn.ModuleList([
            CausalBlock(d_model) for _ in range(2)
        ])
        self.output = nn.Linear(d_model, vocab_size)

    def forward(self, token_ids):
        x = self.position(self.embedding(token_ids))
        for block in self.blocks:
            x = block(x)
        return self.output(x)


lm_model = DecoderOnlyLM(len(lm_vocab)).to(device)
optimizer = torch.optim.Adam(lm_model.parameters(), lr=0.01)

# Teacher forcing: predict the next token
lm_inputs = lm_data[:, :-1]
lm_targets = lm_data[:, 1:]

for epoch in range(301):
    lm_model.train()
    optimizer.zero_grad()

    logits = lm_model(lm_inputs)
    loss = F.cross_entropy(
        logits.reshape(-1, len(lm_vocab)),
        lm_targets.reshape(-1)
    )

    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print(f"Epoch {epoch:3d} | Loss: {loss.item():.4f}")


# Test: generate the continuation of "i like"
def complete_text(model, prompt, max_new_tokens=4):
    model.eval()
    ids = [lm_vocab["<SOS>"]] + encode(prompt, lm_vocab)
    generated = ids.copy()

    with torch.no_grad():
        for _ in range(max_new_tokens):
            input_ids = torch.tensor(
                [generated], dtype=torch.long, device=device
            )
            logits = model(input_ids)
            next_id = logits[0, -1].argmax().item()

            if next_id == lm_vocab["<EOS>"]:
                break

            generated.append(next_id)

    words = [
        id_to_word[i] for i in generated
        if id_to_word[i] not in ("<SOS>", "<EOS>", "<PAD>")
    ]
    return " ".join(words)


print("\nGenerated:", complete_text(lm_model, "i like"))
print("Generated:", complete_text(lm_model, "you love"))


Epoch   0 | Loss: 2.8562
Epoch  50 | Loss: 0.4497
Epoch 100 | Loss: 0.4488
Epoch 150 | Loss: 0.4485
Epoch 200 | Loss: 0.4485
Epoch 250 | Loss: 0.4483
Epoch 300 | Loss: 0.4482

Generated: i like cats
Generated: you love books


Training an Encoder-Decoder Architecture for Machine Translation

In [5]:

translation_pairs = [
    ("i sleep here", "je dors ici"),
    ("you sleep here", "tu dors ici"),
    ("i run fast", "je cours vite"),
    ("you run fast", "tu cours vite"),
]

src_special = ["<PAD>", "<SOS>", "<EOS>", "<UNK>"]
tgt_special = ["<PAD>", "<SOS>", "<EOS>", "<UNK>"]

src_vocab = make_vocab(
    [src for src, tgt in translation_pairs], src_special
)
tgt_vocab = make_vocab(
    [tgt for src, tgt in translation_pairs], tgt_special
)

reverse_tgt_vocab = {i: w for w, i in tgt_vocab.items()}


def encode_source(sentence):
    return [
        src_vocab["<SOS>"],
        *encode(sentence, src_vocab),
        src_vocab["<EOS>"]
    ]


def encode_target(sentence):
    return [
        tgt_vocab["<SOS>"],
        *encode(sentence, tgt_vocab),
        tgt_vocab["<EOS>"]
    ]


src_data = torch.tensor(
    [encode_source(src) for src, tgt in translation_pairs],
    dtype=torch.long,
    device=device
)

tgt_data = torch.tensor(
    [encode_target(tgt) for src, tgt in translation_pairs],
    dtype=torch.long,
    device=device
)


class EncoderDecoderTransformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size):
        super().__init__()
        d_model = 32

        self.src_embedding = nn.Embedding(src_vocab_size, d_model)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model)

        self.src_position = PositionalEncoding(d_model)
        self.tgt_position = PositionalEncoding(d_model)

        self.encoder = nn.ModuleList([
            EncoderBlock(d_model) for _ in range(2)
        ])
        self.decoder = nn.ModuleList([
            DecoderBlock(d_model) for _ in range(2)
        ])

        self.output = nn.Linear(d_model, tgt_vocab_size)

    def encode(self, source_ids):
        x = self.src_position(self.src_embedding(source_ids))
        for block in self.encoder:
            x = block(x)
        return x

    def decode(self, target_ids, encoder_output):
        x = self.tgt_position(self.tgt_embedding(target_ids))
        mask = causal_mask(target_ids.size(1), target_ids.device)

        for block in self.decoder:
            x = block(x, encoder_output, mask)

        return self.output(x)

    def forward(self, source_ids, target_ids):
        encoder_output = self.encode(source_ids)
        return self.decode(target_ids, encoder_output)


seq2seq_model = EncoderDecoderTransformer(
    len(src_vocab), len(tgt_vocab)
).to(device)

optimizer = torch.optim.Adam(seq2seq_model.parameters(), lr=0.01)

# Decoder input starts with SOS; labels are shifted one position
decoder_inputs = tgt_data[:, :-1]
decoder_targets = tgt_data[:, 1:]

for epoch in range(501):
    seq2seq_model.train()
    optimizer.zero_grad()

    logits = seq2seq_model(src_data, decoder_inputs)

    loss = F.cross_entropy(
        logits.reshape(-1, len(tgt_vocab)),
        decoder_targets.reshape(-1)
    )

    loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        print(f"Epoch {epoch:3d} | Loss: {loss.item():.4f}")


# Greedy decoding: generate one French token at a time
def translate_toy(model, sentence, max_new_tokens=6):
    model.eval()

    source_ids = torch.tensor(
        [encode_source(sentence)], dtype=torch.long, device=device
    )
    generated = [tgt_vocab["<SOS>"]]

    with torch.no_grad():
        encoder_output = model.encode(source_ids)

        for _ in range(max_new_tokens):
            target_ids = torch.tensor(
                [generated], dtype=torch.long, device=device
            )
            logits = model.decode(target_ids, encoder_output)
            next_id = logits[0, -1].argmax().item()

            if next_id == tgt_vocab["<EOS>"]:
                break

            generated.append(next_id)

    return " ".join(
        reverse_tgt_vocab[i]
        for i in generated[1:]
        if reverse_tgt_vocab[i] not in ("<PAD>", "<SOS>", "<EOS>")
    )


for source, expected in translation_pairs:
    prediction = translate_toy(seq2seq_model, source)
    print(f"English:  {source}")
    print(f"Expected: {expected}")
    print(f"Predicted: {prediction}\n")


Epoch   0 | Loss: 2.5934
Epoch 100 | Loss: 0.0008
Epoch 200 | Loss: 0.0004
Epoch 300 | Loss: 0.0002
Epoch 400 | Loss: 0.0002
Epoch 500 | Loss: 0.0001
English:  i sleep here
Expected: je dors ici
Predicted: je dors ici

English:  you sleep here
Expected: tu dors ici
Predicted: tu dors ici

English:  i run fast
Expected: je cours vite
Predicted: je cours vite

English:  you run fast
Expected: tu cours vite
Predicted: tu cours vite



Compare the results

In [6]:

print("=== ARCHITECTURE COMPARISON ===")

print(
    "Encoder-only parameters:",
    sum(p.numel() for p in encoder_model.parameters())
)

print(
    "Decoder-only parameters:",
    sum(p.numel() for p in lm_model.parameters())
)

print(
    "Encoder-decoder parameters:",
    sum(p.numel() for p in seq2seq_model.parameters())
)


=== ARCHITECTURE COMPARISON ===
Encoder-only parameters: 17538
Decoder-only parameters: 18063
Encoder-decoder parameters: 43722


In [7]:

encoder_model.eval()

new_sentences = ["this film is good", "this movie is awful"]
new_X = torch.tensor(
    [encode(s, vocab) for s in new_sentences],
    dtype=torch.long,
    device=device
)

with torch.no_grad():
    predictions = encoder_model(new_X).argmax(dim=-1).tolist()

for sentence, prediction in zip(new_sentences, predictions):
    print(f"{sentence} -> {class_names[prediction]}")


this film is good -> positive
this movie is awful -> negative
